In [28]:
import numpy as np
import matplotlib.pyplot as plt

import scipy.optimize as fit

In [29]:
FILE_PATH = input("Input the path to the data to read").strip()
# FILE_PATH = "data/BigArm1.csv"
BIG_ARM = "Big" in FILE_PATH

# OUTPUT_PATH = input("Input the file to write to").strip()
OUTPUT_PATH = FILE_PATH

In [30]:
data = np.genfromtxt(FILE_PATH, delimiter=',', skip_header=True)

In [31]:
def fit_derivative_line(x, x_err, t, index, averaging):
    t_arr = []
    x_arr = []
    x_err_arr = []
    for i in range(averaging):
        j = i + index
        if 0 < j and j < len(x):
            x_arr.append(x[j])
            x_err_arr.append(x_err[j])
            t_arr.append(t[j])

        k = i - index
        if 0 < k and k < len(x):
            x_arr.append(x[k])
            x_err_arr.append(x_err[k])
            t_arr.append(t[k])

    def lin(x, a, b): return a * x + b

    (a, b), pcov = fit.curve_fit(lin, t_arr, x_arr, sigma=x_err_arr, absolute_sigma=True)
    (a_err, b_err) = np.sqrt(np.diag(pcov))

    return (a, a_err), (b, b_err)

def data_derivative(x, x_err, t, averaging):
    d = []
    for i in range(len(x) - 1):
        a, b = fit_derivative_line(x, x_err, t, i, averaging)
        d.append([t[i], a[0], a[1]])
        # print(d)

    return np.array(d)

In [32]:
def single_energy(theta, theta_d, I, m, g, rcm):
    T = 0.5 * I[0] * theta_d[0]**2
    V = -m[0]*g[0]*rcm[0]*np.cos(theta[0])

    T_err = np.hypot(I[1] * 0.5*theta_d[0]**2, theta_d[1] * I[0] * theta_d[0])
    V_err = np.sqrt(
        (m[1] * g[0]*rcm[0]*np.cos(theta[0]))**2 +
        (g[1] * m[0]*rcm[0]*np.cos(theta[0]))**2 +
        (rcm[1] * m[0]*g[0]*np.cos(theta[0]))**2 +
        (theta[1] * m[0]*g[0]*rcm[0]*np.sin(theta[0]))**2
    )

    return (T, T_err), (V, V_err)

In [33]:
t = data[:, 0]

theta = data[:, 9]
theta_err = data[:, 11]

theta_d_data = data_derivative(data[:, 9], data[:, 11], data[:, 0], 5)
theta_d = theta_d_data[:, 1]
theta_d_err = theta_d_data[:, 2]

T = []; V = []

if BIG_ARM:
    T, V = single_energy(
        (theta[1:], theta_err[1:]),
        (theta_d, theta_d_err),
        (0.00216, 0),
        (108.5e-3, 0),
        (9.81, 0),
        (11.7e-2, 0)
    )
else: 
    T, V = single_energy(
        (theta[1:], theta_err[1:]),
        (theta_d, theta_d_err),
        (0.00103, 0),
        (86e-3, 0),
        (9.81, 0),
        (9.225e-2, 0)
    )

C:\Users\willd\AppData\Local\Temp\ipykernel_29052\1623177283.py:20: OptimizeWarning: Covariance of the parameters could not be estimated
  (a, b), pcov = fit.curve_fit(lin, t_arr, x_arr, sigma=x_err_arr, absolute_sigma=True)


In [38]:
# t_rel = t - t[0]
# skip = 2
# plt.errorbar(t_rel[::skip], T[0][::skip], T[1][::skip], fmt='o')
# plt.errorbar(t_rel[::skip], V[0][::skip], V[1][::skip], fmt='o')
# # plt.plot(t[1:]-t[0], E, 'x')
# plt.xlim(13, 18)

In [35]:
E_data = np.vstack(
    (
        T[0],
        T[1],
        V[0],
        V[1]
    )
).transpose()
E_data = np.vstack((E_data, E_data[-1]))

In [36]:
out = np.hstack(
    (data, E_data)
)

np.savetxt(OUTPUT_PATH, out, delimiter=",", fmt="%f", header="time (s), x1 (pixels), y1 (pixels), x1 error (pixels), y1 error (pixels), x2 (pixels), y2 (pixels), x2 error (pixels), y2 error (pixels), angle1 (the smaller arm) (rad), angle2 (the larger arm) (rad), angle1 error (rad), angle2 error (rad), kinetic energy (J), kinetic energy error (J), potential energy (J), potential energy error (J)")